In [1]:
import pandas as pd 
import numpy as np
import statsmodels.api as sm
import glob
import os

# ========== 1. 读取并合并日度 CSV 数据 ==========
folder_path = r"D:\桌面\绿色货币\数据总库\R数据抓取加密货币数据"  # 数据文件夹路径
file_paths = sorted(glob.glob(os.path.join(folder_path, "crypto_full_20*_Daily_data.csv")))
df_daily = pd.concat((pd.read_csv(file) for file in file_paths), ignore_index=True)

# ========== 2. 定义聚合日度数据为周度数据的函数 ==========
def aggregate_daily_to_weekly(df):
    # 1) 时间转换 & 筛选日期
    df = df.rename(columns={"time_open": "Date"})
    df["Date"] = pd.to_datetime(df["Date"])
    # 如有需要可筛选时间范围，例如：
    # df = df.query('"2014-01-01" <= Date <= "2020-07-31"')
    
    # 2) 使用 ISO 8601 周规则：获取 ISO 年和 ISO 周
    iso = df["Date"].dt.isocalendar()
    df["iso_year"] = iso.year
    df["iso_week"] = iso.week

    # 3) 数据清洗：剔除缺失及异常值；同时对市值筛选（大于 1,000,000）
    df = df.dropna()
    df = df[(df["volume"] > 0) & (df["close"] > 0)]
    df = df.dropna(subset=["market_cap"])
    df = df[df["market_cap"] > 1_000_000]
    
    # 4) 计算每日价格成交额及其对数
    df["price_vol"] = df["close"] * df["volume"]
    df["log_price_vol"] = np.log(df["price_vol"])
    
    # 5) 按币种、ISO年和ISO周聚合（聚合规则与论文一致）
    agg_dict = {
        "Date": "max",          # 每只币当周最后交易日
        "close": "last",        # 本周最后一天收盘价
        "high": "max",          # 本周最高价
        "market_cap": "last",   # 本周最后一天市值
        "volume": "mean",       # 本周平均交易量
        "price_vol": "mean",    # 本周平均成交额
        "log_price_vol": "std"  # 本周对数成交额样本标准差（ddof=1）
    }
    weekly = df.groupby(["id", "iso_year", "iso_week"], as_index=False).agg(agg_dict)
    
    # 6) 重命名聚合列 & 生成对数特征
    weekly.rename(columns={
        "Date": "Date_max",
        "close": "close_last",
        "high": "high_max",
        "market_cap": "market_cap_last",
        "volume": "volume_mean",
        "price_vol": "price_vol_mean",
        "log_price_vol": "log_price_vol_std"
    }, inplace=True)
    weekly["MCAP"] = np.log(weekly["market_cap_last"])
    weekly["PRC"] = np.log(weekly["close_last"])
    weekly["MAXDPRC"] = np.log(weekly["high_max"])
    weekly["PRCVOL"] = np.log(weekly["price_vol_mean"])
    weekly.rename(columns={"log_price_vol_std": "STDPRCVOL"}, inplace=True)
    
    # 7) 计算周度对数收益率（按每只币的收盘价差分）
    weekly.sort_values(["id", "Date_max"], inplace=True)
    weekly["rtn"] = weekly.groupby("id")["close_last"].transform(lambda x: np.log(x).diff())
    
    # 8) 删除因差分产生的缺失值
    weekly.dropna(subset=["rtn"], inplace=True)
    
    # 9) 统一每周日期：使用 ISO 年、ISO 周及星期日作为代表日期
    weekly["Date"] = weekly.apply(lambda row: pd.Timestamp.fromisocalendar(int(row["iso_year"]), int(row["iso_week"]), 7), axis=1)
    
    return weekly

# ========== 3. 调用函数，得到周度数据 ==========
df_weekly = aggregate_daily_to_weekly(df_daily)

# 添加 "year" 列，从 Date 列中提取年份（用于后续按年份汇总）
df_weekly["year"] = df_weekly["Date"].dt.year

# ========== 4. 按年份汇总，生成统计表 (Panel A) ==========
# 只保留 2014~2020 年
df_weekly = df_weekly[df_weekly["year"].between(2014, 2024)]

# 按年份分组，计算需要的统计量：
#   Number：该年出现过的独立币种数
#   Market Cap (mil)：market_cap_last 的均值和中位数 (单位: 百万美元)
#   Volume (thous)：volume_mean 的均值和中位数 (单位: 千美元)
yearly_stats = (
    df_weekly
    .groupby("year", as_index=False)
    .agg(
        Number             = ("id", "nunique"),
        MarketCapMean      = ("market_cap_last", lambda x: x.mean() / 1e6),
        MarketCapMedian    = ("market_cap_last", lambda x: x.median() / 1e6),
        VolumeMean         = ("volume_mean", lambda x: x.mean() / 1e3),
        VolumeMedian       = ("volume_mean", lambda x: x.median() / 1e3)
    )
    .sort_values("year")
)

# ========== 5. 计算 Full (2014-2020) 汇总行 ==========
mask_2014_2020 = df_weekly["year"].between(2014, 2024)
full_stats = {
    "year"            : "Full",
    "Number"          : df_weekly.loc[mask_2014_2020, "id"].nunique(),
    "MarketCapMean"   : df_weekly.loc[mask_2014_2020, "market_cap_last"].mean() / 1e6,
    "MarketCapMedian" : df_weekly.loc[mask_2014_2020, "market_cap_last"].median() / 1e6,
    "VolumeMean"      : df_weekly.loc[mask_2014_2020, "volume_mean"].mean() / 1e3,
    "VolumeMedian"    : df_weekly.loc[mask_2014_2020, "volume_mean"].median() / 1e3
}
full_stats_df = pd.DataFrame([full_stats])

# 合并到最终结果
final_stats = pd.concat([yearly_stats, full_stats_df], ignore_index=True)

# ========== 6. 重命名列并控制小数位 ==========
final_stats = final_stats.rename(columns={
    "year": "Year",
    "MarketCapMean"  : "Market Cap (mil) Mean",
    "MarketCapMedian": "Market Cap (mil) Median",
    "VolumeMean"     : "Volume (thous) Mean",
    "VolumeMedian"   : "Volume (thous) Median"
})

# 选择性：保留两位小数
final_stats["Market Cap (mil) Mean"]   = final_stats["Market Cap (mil) Mean"].round(2)
final_stats["Market Cap (mil) Median"] = final_stats["Market Cap (mil) Median"].round(2)
final_stats["Volume (thous) Mean"]     = final_stats["Volume (thous) Mean"].round(2)
final_stats["Volume (thous) Median"]   = final_stats["Volume (thous) Median"].round(2)

# ========== 7. 查看最终结果 ==========
print(final_stats)


    Year  Number  Market Cap (mil) Mean  Market Cap (mil) Median  \
0   2014      44                 328.43                     6.33   
1   2015      38                 191.05                     5.24   
2   2016      79                 241.03                     5.20   
3   2017     276                 905.51                    18.91   
4   2018     516                 817.22                    21.16   
5   2019     593                 479.28                     9.06   
6   2020     843                 557.97                    10.29   
7   2021    1556                1794.18                    28.13   
8   2022    1665                1078.39                    17.37   
9   2023    1826                 937.22                    15.68   
10  2024    2634                1445.70                    18.23   
11  Full    3273                1145.57                    16.76   

    Volume (thous) Mean  Volume (thous) Median  
0               1547.34                  50.89  
1               1

In [2]:
final_stats.to_csv('2014-2024 final_stats.csv')